# 00 — Smoke test

Run this first (Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all).

It checks, in about 20 minutes, that every serving setup the project needs can
start on this GPU and handle a tool-calling request:

1. vLLM with the small model (Qwen3-0.6B)
2. vLLM with the large model (Qwen3-8B, 4-bit AWQ)
3. vLLM + LMCache
4. vLLM + LMCache, older in-process integration (fallback only)

A setup that fails does not stop the notebook; the last cells show a summary and
download a zip with the logs.

In [ ]:
# --- Setup: get the code and install the agent-side packages ---
REPO_URL = "https://github.com/sauerpatchkid/Portfolio_Assistant.git"
REPO_DIR = "/content/portfolio-assistant"

import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Serving environment: vLLM + LMCache in their own virtualenv ---
# The server runs as a separate process, so it gets a separate environment.
# That keeps its packages from clashing with the ones Colab ships in the notebook.
VENV = "/content/venv-vllm"

!pip install -q uv
if not os.path.exists(VENV):
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python vllm lmcache
os.environ["SERVING_VENV"] = VENV

# Record the exact versions so the results can be reproduced.
!uv pip freeze --python {VENV}/bin/python | grep -iE "^(vllm|lmcache|torch|transformers)==" | tee results/versions_vllm.txt

In [ ]:
from serving.launch import start_server, stop_server, run_command, tail

SMALL_MODEL = "Qwen/Qwen3-0.6B"
LARGE_MODEL = "Qwen/Qwen3-8B-AWQ"


def smoke(config, model, name, extra_commands=()):
    """Start one serving setup, run the smoke check against it, stop it. Never raises."""
    server = None
    try:
        server = start_server(config, model, log_name=name)
        run_command(f"python -m scripts.smoke_check --name {name} --model {model}")
        for command in extra_commands:
            run_command(command)
    except Exception as e:
        print(f"FAILED: {name}: {e}")
    finally:
        stop_server(server)

## 1. vLLM + Qwen3-0.6B

In [ ]:
smoke("vllm", SMALL_MODEL, "smoke_vllm_small")

## 2. vLLM + Qwen3-8B (AWQ)

Also runs 20 eval questions and a short time-to-first-token pass, to see how
long the full runs will take.

In [ ]:
smoke("vllm", LARGE_MODEL, "smoke_vllm_large", extra_commands=[
    f"python -m eval.run_eval --name smoke_eval_c8 --model {LARGE_MODEL} --concurrency 8 --limit 20",
    f"python -m bench.ttft --name smoke_ttft --model {LARGE_MODEL} --limit 5",
])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 3. vLLM + LMCache

In [ ]:
smoke("vllm_lmcache", LARGE_MODEL, "smoke_lmcache")

# LMCache activity in the server log (stored / retrieved KV cache) and what reached the disk tier
!grep -iE "lmcache" results/logs/smoke_lmcache.log | tail -n 25
!du -sh /content/lmcache_disk 2>/dev/null

## 4. vLLM + LMCache, in-process (fallback)

In [ ]:
smoke("vllm_lmcache_inprocess", LARGE_MODEL, "smoke_lmcache_inprocess")
!grep -iE "lmcache" results/logs/smoke_lmcache_inprocess.log | tail -n 25

## Summary

In [ ]:
import glob, json

for path in sorted(glob.glob("results/smoke/*.json")):
    report = json.load(open(path))
    checks = report["checks"]
    passed = sum(c["passed"] for c in checks.values())
    print(f"{os.path.basename(path):<34} {passed}/{len(checks)} passed  {report.get('error', '')}")
    for name, c in checks.items():
        if not c["passed"]:
            print(f"    FAIL {name}: {c['detail'][:200]}")

for name in ["smoke_vllm_small", "smoke_vllm_large", "smoke_lmcache", "smoke_lmcache_inprocess"]:
    if not os.path.exists(f"results/smoke/{name}.json"):
        print(f"{name}: did not start, see results/logs/{name}.log")

In [ ]:
# --- Download the results ---
!zip -qr /content/results_00_smoke.zip results
from google.colab import files
files.download("/content/results_00_smoke.zip")